In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.impute import SimpleImputer

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Q3_data_path = os.path.join(path, 'Q3_data.csv')
df_Q3_data = pd.read_csv(Q3_data_path)

In [ ]:
# Task 2: Write your code here:
df_Q3_data.head()

In [ ]:
# Task 3: Write your code here:
df_Q3_data.info()

In [ ]:
# Task 4: Write your code here:
df_Q3_data.describe()

In [ ]:
# Task 1: Write your code here:
# Missing values
print("Missing values:")
print(df_Q3_data.isnull().sum())

num_cols = df_Q3_data.select_dtypes(include=['number']).columns

num_imputer = SimpleImputer(strategy='median')

# use .copy() to keep original df safe
df_clean = df_Q3_data.copy()

# Fill Missing Values
df_clean[num_cols] = num_imputer.fit_transform(df_Q3_data[num_cols])

print(f"Missing values after cleaning: {df_clean.isnull().sum().sum()}")
print(df_clean.isnull().sum())

In [ ]:
# Task 2: Write your code here:
# Duplicate Check

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_Q3_data)

In [ ]:
# Task 3: Write your code here:
# there are no categorical variables, there is no need

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

print('data before scaling:\n', df_Q3_data) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(df_Q3_data) # Apply fit_transform

print('\nData after scaling:\n', data_standard_scaled) #show after scaling

In [ ]:
# Task 5: Write your code here:
import seaborn as sns

TARGET_COL = 'Target'

plt.figure(figsize=(8, 5))
if df_clean[TARGET_COL].dtype == 'object' or df_clean[TARGET_COL].nunique() < 10:
    # Classification Problem
    sns.countplot(x=TARGET_COL, data=df_clean, palette='viridis')
    plt.title(f"Target Distribution: {TARGET_COL} (Classification)")
    plt.ylabel("Count")
else:
    # Regression Problem
    sns.histplot(df_clean[TARGET_COL], kde=True, color='blue')
    plt.title(f"Target Distribution: {TARGET_COL} (Regression)")
    plt.xlabel("Value")
plt.show()

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split

# --- 1. Separate Features (X) and Target (y) ---
X = df_Q3_data.drop(columns=[TARGET_COL])
y = df_Q3_data[TARGET_COL]

# --- 2. Split Data ---
# Logic: If Classification -> Use Stratify. If Regression -> No Stratify.
is_classification = (y.dtype == 'object') or (y.nunique() < 20)

if is_classification:
    print("Detected Classification Task -> Using Stratified Split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
else:
    print("Detected Regression Task -> Using Random Split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print(f"Train Shape: {X_train.shape}")
print(f"Test Shape:  {X_test.shape}")


In [ ]:
# Task 2,3,4,5: Write your code here:
!pip install catboost
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = {}
model['CatBoost'] = CatBoostRegressor(verbose=0, random_state=42) # verbose=0 silences output

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Predictions and metrics
y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Target']))

In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: